# SpaceX Falcon 9 — Exploratory Data Analysis with SQL

**Objective:** load the SpaceX launch dataset into a SQLite database and answer the course questions with SQL queries (launch sites, payloads, landing outcomes, boosters and time analysis).

In [13]:
import piplite
await piplite.install(['numpy', 'pandas'])
try:
    await piplite.install('sqlite3')   # SQLite module of Pyodide
except Exception as e:
    print(e)

Can't fetch metadata for 'sqlite3'. Please make sure you have entered a correct package name and correctly specified index_urls (if you changed them).


In [14]:
import pandas as pd
import sqlite3
pd.set_option('display.max_colwidth', None)

In [15]:
from js import fetch
import io, json

async def get_text(url):
    resp = await fetch(url)
    if not resp.ok:
        raise Exception(f"HTTP {resp.status} for {url}")
    return await resp.text()

async def get_json(url):
    return json.loads(await get_text(url))

async def get_csv(url):
    return pd.read_csv(io.StringIO(await get_text(url)))

## Load the dataset into a SQLite table

In [16]:
df = await get_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")

# Standard column names
df.columns = ['Date', 'Time_UTC', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_',
              'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']

# Dates stored as ISO text (YYYY-MM-DD) so they can be filtered and sorted in SQL
iso = df['Date'].astype(str).str.match(r'^\d{4}-')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=not iso.all()).dt.strftime('%Y-%m-%d')

con = sqlite3.connect("my_data1.db")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False)
con.execute("DROP TABLE IF EXISTS SPACEXTABLE")
con.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")

def sql(query):
    return pd.read_sql_query(query, con)

sql("SELECT COUNT(*) AS total_records FROM SPACEXTABLE")

<class 'pandas.errors.DatabaseError'>: Execution failed

## Task 1: Names of the unique launch sites

In [ ]:
sql("SELECT DISTINCT Launch_Site FROM SPACEXTABLE")

## Task 2: 5 records where launch sites begin with the string 'CCA'

In [ ]:
sql("SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

## Task 3: Total payload mass carried by boosters launched by NASA (CRS)

In [ ]:
sql("""SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass_kg
       FROM SPACEXTABLE
       WHERE Customer = 'NASA (CRS)'""")

## Task 4: Average payload mass carried by booster version F9 v1.1

In [ ]:
sql("""SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_mass_kg
       FROM SPACEXTABLE
       WHERE Booster_Version = 'F9 v1.1'""")

## Task 5: Date of the first successful landing outcome on a ground pad

In [ ]:
sql("""SELECT MIN(Date) AS first_successful_ground_pad_landing
       FROM SPACEXTABLE
       WHERE Landing_Outcome = 'Success (ground pad)'""")

## Task 6: Boosters with success on a drone ship and payload mass greater than 4000 but less than 6000

In [ ]:
sql("""SELECT Booster_Version
       FROM SPACEXTABLE
       WHERE Landing_Outcome = 'Success (drone ship)'
         AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000""")

## Task 7: Total number of successful and failed mission outcomes

In [ ]:
sql("""SELECT Mission_Outcome, COUNT(*) AS total
       FROM SPACEXTABLE
       GROUP BY Mission_Outcome""")

## Task 8: Booster versions that carried the maximum payload mass (subquery)

In [ ]:
sql("""SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_
       FROM SPACEXTABLE
       WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)""")

## Task 9: Failed drone-ship landings in 2015 — month, booster version and launch site

In [ ]:
sql("""SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
       FROM SPACEXTABLE
       WHERE substr(Date, 1, 4) = '2015'
         AND Landing_Outcome = 'Failure (drone ship)'""")

## Task 10: Rank the count of landing outcomes between 2010-06-04 and 2017-03-20, in descending order

In [ ]:
sql("""SELECT Landing_Outcome, COUNT(*) AS total
       FROM SPACEXTABLE
       WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
       GROUP BY Landing_Outcome
       ORDER BY total DESC""")

## Author

Jhoselyn Vera